## Variable Diagnostic for Youth Screentime Questionaire variables

Looking for variables that can be used best longitudinally, have the least amount of missing data, and is a good descriptor of overall screentime 

Finding what variables exist for youth screentime questionaire 

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Load wide-format dataset
df = pd.read_csv('/Users/jasmine/Downloads/Thesis-Lab/dataset.tsv', sep='\t', low_memory=False)

# Find all youth screentime variables
stq_vars = [c for c in df.columns if c.startswith('nt_y_stq__')]

print(f"Found {len(stq_vars)} nt_y_stq variables:\n")
for v in stq_vars:
    print(v)

dataset configuration

In [ ]:
# Configuration for variable diagnostics
SUBJ_COL = 'participant_id'
TP_COL   = 'session_id'
TABLE    = 'nt_y_stq__'

TIMEPOINTS = [
    'ses-000A', 'ses-01A', 'ses-02A',
    'ses-03A',  'ses-04A', 'ses-05A', 'ses-06A'
]

# Subdomain filters (for composite ranking bonus)
SUBDOMAINS = {
    'screen_weekday': 'nt_y_stq__screen__wkdy',
    'screen_weekend': 'nt_y_stq__screen__wknd',
    'sleep':          'nt_y_stq__sleep',
    'social_media':   'nt_y_stq__socmed',
}

# Summary/total score priority flags (for composite ranking bonus)
SUMMARY_KEYWORDS = ['sum', 'tot', 'nm']


Loading the Data 

In [ ]:
df = pd.read_csv('/Users/jasmine/Downloads/Thesis-Lab/dataset.tsv', sep='\t', low_memory=False)

# Get all stq variables
stq_vars = [c for c in df.columns if c.startswith(TABLE)]

# Tag each variable with its subdomain
def get_subdomain(var):
    for label, prefix in SUBDOMAINS.items():
        if var.startswith(prefix):
            return label
    return 'other'

# Create a DataFrame to hold variable metadata
var_meta = pd.DataFrame({
    'variable': stq_vars,
    'subdomain': [get_subdomain(v) for v in stq_vars],
    'is_summary': [any(k in v for k in SUMMARY_KEYWORDS) for v in stq_vars]
})

print(f"Total nt_y_stq variables: {len(stq_vars)}")
print(var_meta['subdomain'].value_counts())
print(f"\nSummary/total score variables: {var_meta['is_summary'].sum()}")
print(var_meta[var_meta['is_summary']]['variable'].tolist())

Diagnostic for missingness

In [ ]:
print("\n\nRunning missingness diagnostic...")

# Calculate missingness for each variable at each timepoint
miss_records = []
for var in stq_vars:
    for tp, grp in df.groupby(TP_COL):
        if tp not in TIMEPOINTS:
            continue
        n_total   = len(grp)
        n_missing = grp[var].isna().sum()
        miss_records.append({
            'variable':    var,
            'subdomain':   get_subdomain(var),
            'is_summary':  any(k in var for k in SUMMARY_KEYWORDS),
            'timepoint':   tp,
            'n_total':     n_total,
            'n_missing':   n_missing,
            'pct_missing': round(n_missing / n_total * 100, 2)
        })

miss_df      = pd.DataFrame(miss_records)
miss_summary = (miss_df.groupby(['variable', 'subdomain', 'is_summary'])['pct_missing']
                .mean().reset_index()
                .rename(columns={'pct_missing': 'mean_pct_missing'})
                .sort_values('mean_pct_missing'))

print("\n── Top 20 variables by lowest missingness ──")
print(miss_summary.head(20).to_string(index=False))

Longitudinal diagnostic 

In [ ]:
print("\n\nRunning longitudinal coverage diagnostic...")

# Calculate longitudinal coverage for each variable
long_records = []
# For each variable, count how many subjects have non-missing data at 1+ waves, 2+ waves, and all waves
for var in stq_vars:
    sub_df       = df[df[TP_COL].isin(TIMEPOINTS)][[SUBJ_COL, TP_COL, var]]
    sub_nonmiss  = sub_df.dropna(subset=[var])
    wave_counts  = sub_nonmiss.groupby(SUBJ_COL)[TP_COL].nunique()

    long_records.append({
        'variable':              var,
        'subdomain':             get_subdomain(var),
        'is_summary':            any(k in var for k in SUMMARY_KEYWORDS),
        'n_subjects_1+_waves':   (wave_counts >= 1).sum(),
        'n_subjects_2+_waves':   (wave_counts >= 2).sum(),
        'n_subjects_complete':   (wave_counts == len(TIMEPOINTS)).sum(),
        'mean_waves_per_subject': round(wave_counts.mean(), 2)
    })

long_df = pd.DataFrame(long_records).sort_values('n_subjects_2+_waves', ascending=False)
print("\n── Top 20 variables by longitudinal coverage ──")
print(long_df.head(20).to_string(index=False))

Variance diagnostic

In [ ]:
print("\n\nRunning variance diagnostic...")

# Calculate variance for each variable at each timepoint, then average across timepoints
var_records = []
for var in stq_vars:
    tp_variances = []
    for tp, grp in df.groupby(TP_COL):
        if tp not in TIMEPOINTS:
            continue
        v = grp[var].var()
        if not np.isnan(v):
            tp_variances.append(v)
    var_records.append({
        'variable':      var,
        'subdomain':     get_subdomain(var),
        'is_summary':    any(k in var for k in SUMMARY_KEYWORDS),
        'mean_variance': round(np.nanmean(tp_variances), 4) if tp_variances else np.nan
    })

# Create DataFrame and sort by mean variance
var_df = pd.DataFrame(var_records).sort_values('mean_variance', ascending=False)
print("\n── Top 20 variables by variance ──")
print(var_df.head(20).to_string(index=False))

composite ranking

In [ ]:
print("\n\nBuilding composite ranking...")

# Normalize each metric to 0-1 (higher is better), then combine with weights
scaler  = MinMaxScaler()
ranking = miss_summary.merge(long_df[['variable', 'n_subjects_2+_waves', 'mean_waves_per_subject']], on='variable', how='left')
ranking = ranking.merge(var_df[['variable', 'mean_variance']], on='variable', how='left')

ranking['miss_score'] = 1 - scaler.fit_transform(ranking[['mean_pct_missing']])
ranking['long_score'] = scaler.fit_transform(ranking[['n_subjects_2+_waves']])
ranking['var_score']  = scaler.fit_transform(ranking[['mean_variance']])

# Summary score bonus (+0.1) to prioritize aggregate variables per your goal
ranking['summary_bonus'] = ranking['is_summary'].astype(float) * 0.10

ranking['composite_score'] = (
    ranking['miss_score']    * 0.30 +
    ranking['long_score']    * 0.30 +
    ranking['var_score']     * 0.20 +
    ranking['summary_bonus']
)

final = ranking.sort_values('composite_score', ascending=False)

print("\n══ FINAL VARIABLE RANKING (Top 30) ══")
print(final[['variable', 'subdomain', 'is_summary', 'composite_score',
             'mean_pct_missing', 'n_subjects_2+_waves',
             'mean_variance']].head(30).to_string(index=False))

# Save full ranking
final.to_csv('stq_variable_diagnostic.csv', index=False)
print("\n✓ Full ranking saved to stq_variable_diagnostic.csv")

missingness heatmap

In [ ]:
# ── HEATMAP: Missingness by variable × timepoint ─────────────────────────────
summary_vars = var_meta[var_meta['is_summary']]['variable'].tolist()

# Create pivot table for heatmap - this will automatically exclude any timepoints that don't have data for these variables
pivot = miss_df[miss_df['variable'].isin(summary_vars)].pivot_table(
    index='variable', columns='timepoint', values='pct_missing'
)

# Only keep timepoints that actually exist in the pivot columns
available_timepoints = [tp for tp in TIMEPOINTS if tp in pivot.columns]
pivot = pivot[available_timepoints]

print(f"Timepoints present in heatmap: {available_timepoints}")
print(f"Summary variables in heatmap: {len(pivot)}")

plt.figure(figsize=(12, max(6, len(pivot) * 0.4)))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, cbar_kws={'label': '% Missing'})
plt.title('Missingness Heatmap — Summary Score Variables (nt_y_stq)')
plt.tight_layout()
plt.savefig('stq_missingness_heatmap.png', dpi=150)
plt.show()
print("✓ Heatmap saved to stq_missingness_heatmap.png") 